# PCA Phase Coordinates Demo

This notebook demonstrates `fit_pca_phase_coordinates`, which fits a local PCA plane to each movement cycle and returns per-sample phase, radius, and perpendicular deviation coordinates.

The algorithm is fast and works for 3-D or higher-dimensional data. Phase can be supplied directly or estimated via the Hilbert transform.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from phase_coordinates import fit_pca_phase_coordinates, reconstruct_phase_coordinates

In [ ]:
rng = np.random.default_rng(0)
n = 300
t = np.linspace(0, 6 * np.pi, n)
tilt = np.pi / 6
X = np.column_stack([np.cos(t), np.sin(t) * np.cos(tilt), np.sin(t) * np.sin(tilt)])
X += rng.normal(0, 0.02, X.shape)
phase = t.copy()
print(f"X shape: {X.shape}, ~3 full cycles")

In [ ]:
samples, cycles, details = fit_pca_phase_coordinates(X, phase=phase)
print("samples shape:", samples.shape)
print("cycles shape:", cycles.shape)
print("algorithm:", details["algorithm"])
samples.head()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
axes[0].plot(samples["phase_in_cycle"])
axes[0].set_ylabel("phase_in_cycle (rad)")
axes[1].plot(samples["radius"])
axes[1].set_ylabel("radius")
axes[2].plot(samples["perp"])
axes[2].set_ylabel("perp")
axes[2].set_xlabel("sample")
plt.tight_layout()
plt.show()

In [ ]:
X_hat = reconstruct_phase_coordinates(samples, cycles)
fitted = ~np.isnan(X_hat[:, 0])
err = np.linalg.norm(X_hat[fitted] - X[fitted], axis=1)
print(f"Max reconstruction error (3-D PCA): {err.max():.2e}")
print("(Near machine precision for 3-D data)")

In [ ]:
print("Fitted cycles:")
print(cycles[["cycle", "n_samples", "radius_mean", "perp_mean", "fit_ok"]])

In [ ]:
cyc0 = details["models"][0]
print("Cycle 0 explained variance:", cyc0["explained_variance_ratio"])
print("Cycle 0 e1 component:", cyc0["components"][0])